In [0]:
%pip install sqlglot -q
%restart_python

In [0]:
import sqlglot
import re
from sqlglot.expressions import Table, CTE
from sqlglot.errors import ErrorLevel
from pyspark.sql.functions import udf, col, explode, split, when, size, element_at, trim, upper,row_number, desc
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql.window import Window


def clean_bo_sql(sql):
    """Remove SAP BO-specific syntax that sqlglot cannot parse."""
    sql = re.sub(r"@Prompt\((?:[^()]*|\([^()]*\))*\)", "'PROMPT_PLACEHOLDER'", sql)
    sql = re.sub(r"@Variable\((?:[^()]*|\([^()]*\))*\)", "'VAR_PLACEHOLDER'", sql)
    sql = re.sub(r'\.\s+', '.', sql)
    return sql

def extract_tables(sql):
    try:
        clean_sql = clean_bo_sql(sql or "")
        tables = set()
        cte_names = set()
        for parsed in sqlglot.parse(clean_sql, read="oracle", error_level=ErrorLevel.IGNORE):
            for cte in parsed.find_all(CTE):
                cte_names.add(cte.alias.lower())
            for t in parsed.find_all(Table):
                catalog = t.catalog
                schema = t.db
                name = t.name
                full_name = ".".join(
                    part for part in [catalog, schema, name] if part
                )
                if full_name.lower() not in cte_names:
                    tables.add(full_name)
        return sorted(tables)
    except Exception as e:
        return []

extract_tables_udf = udf(extract_tables, ArrayType(StringType()))

# Step 1: Get unprocessed Document_Ids using Python set difference (no ANTI JOIN)
# Query 1: Already-processed IDs (~7K distinct values from 170K rows)
processed_ids = set(    row.Document_Id for row in
    spark.sql("SELECT DISTINCT Document_Id FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source").collect())
print(f"Already processed: {len(processed_ids)} documents")

# Query 2: All candidate IDs from source (excluding non-SQL rows)
candidate_ids = [ row.Document_Id for row in
    spark.sql("""
        SELECT DISTINCT Document_Id 
        FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
        WHERE SQL_Query NOT IN ('Error retrieving Query Plan', 'Data Source Type excel not handled for SQL extraction')
    """).collect()]
print(f"Total candidates: {len(candidate_ids)} documents")

batch_size = 1000
total_candidates = len(candidate_ids)
batches = [candidate_ids[i:i+batch_size] for i in range(0, total_candidates, batch_size)]


df_source = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")
w = Window.partitionBy("Document_Id", "Data_Provider_ID", "SQL_Index").orderBy(desc("ingestion_ts"))
df_deduped = df_source.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")

for idx, batch_ids in enumerate(batches):
    document_ids = [doc_id for doc_id in batch_ids if doc_id not in processed_ids]
    print(f"Batch {idx+1}/{len(batches)}: {len(document_ids)} documents to process.")

    if not document_ids:
        print("All documents in this batch already processed!")
        continue
    df_filtered = df_deduped.filter(col("Document_Id").isin(document_ids))

    df_with_tables = df_filtered.withColumn("parsed_tables", extract_tables_udf(col("SQL_Query")))
    df_exploded = df_with_tables.withColumn("table", explode(col("parsed_tables")))

    split_col = split(col("table"), r"\.")
    df_final = df_exploded.withColumn(
        "sql_table", upper(trim(col("table")))
    ).withColumn(
        "table_Name", upper(trim(element_at(split_col, -1)))
    ).withColumn(
        "schema_Name", when(size(split_col) > 1, upper(trim(element_at(split_col, -2))))
    )

    final_df = df_final.select(
        col("Document_Id"),
        col("Document_CUID"),
        col("Document_name"),
        col("Full_path"),
        col("lastAuthor").alias("updated_by"),
        col("Connection_Name").alias("source_DB_connection"),
        col("sql_table"),
        col("table_Name"),
        col("schema_Name")
    ).distinct()

    final_df.write.mode("append").saveAsTable(
        "dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source"
    )
    print(f"Batch {idx+1} written to active_webi_source.")

    spark.sql("OPTIMIZE dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source")
    print("Table optimized after batch.")

print("All batches processed.")